In [ ]:
"""
Section 3.1 simulation + figures
Jointly trainable source-embedding version

This script removes the inconsistency between the theory section and the
simulation section by fitting source-aware models in which the embedding
matrix is trained jointly with the remaining model parameters.

Experiments
-----------
1. Multiclass classification with jointly trainable source embeddings
2. Covariance inflation under source mixing (generative verification of the
   covariance decomposition)
3. Linear regression with jointly trainable source embeddings

Outputs
-------
CSV:
    section_3_1_classification_results.csv
    section_3_1_covariance_results.csv
    section_3_1_ovb_results.csv

Figures:
    figure1.png   # covariance inflation
    figure2.png   # prediction error
    figure3.png   # prediction gain
    figure4.png   # covariance traces
    figure5.png   # coefficient error
    figure6.png   # residual inflation
"""

from dataclasses import dataclass
from typing import List, Dict, Tuple

# make_source_complexity_figures.py

import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------
# Global settings
# ----------------------------

OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Use 100 for final manuscript figures. Use 10-20 for fast debugging.
N_REPS = 100

C = 8          # number of classes
P = 6          # predictor dimension
R0 = 4         # true latent rank in low-rank regime
GAMMA = 1.0
SIGMA_BETA = 0.5

K_GRID = [10, 25, 50, 100]
M_GRID = [2, 4, 8, 16]
NS_GRID = [40, 200]
REGIMES = ["low_rank", "unrestricted"]

N_TEST = 20000
LAMBDA_GRID = [1e-4, 1e-3, 1e-2, 1e-1]

EPOCHS = 100
LR = 3e-2
PATIENCE = 20
VAL_FRAC = 0.2


# ----------------------------
# Utilities
# ----------------------------

def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)


def sample_categorical(prob, rng):
    u = rng.uniform(size=prob.shape[0])
    return (prob.cumsum(axis=1) < u[:, None]).sum(axis=1)


def train_val_split(n, val_frac, rng):
    idx = np.arange(n)
    rng.shuffle(idx)
    n_val = int(np.floor(val_frac * n))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]
    return tr_idx, val_idx


# ----------------------------
# Data generation
# ----------------------------

def generate_parameters(K, regime, rng):
    alpha = np.zeros(C)

    beta = rng.normal(
        loc=0.0,
        scale=SIGMA_BETA,
        size=(C, P)
    )

    if regime == "low_rank":
        U = rng.normal(size=(K, R0))
        V = rng.normal(scale=1.0 / np.sqrt(R0), size=(C, R0))
        Q = GAMMA * U @ V.T

    elif regime == "unrestricted":
        Q = rng.normal(loc=0.0, scale=GAMMA, size=(K, C))

    else:
        raise ValueError("Unknown regime.")

    # Center source effects over sources for identifiability.
    Q = Q - Q.mean(axis=0, keepdims=True)

    return alpha, beta, Q


def sample_dataset(K, n_per_source, alpha, beta, Q, rng, balanced=True):
    if balanced:
        S = np.repeat(np.arange(K), n_per_source)
        rng.shuffle(S)
        n = len(S)
    else:
        n = n_per_source
        S = rng.integers(low=0, high=K, size=n)

    X = rng.normal(size=(n, P))
    logits = alpha[None, :] + X @ beta.T + Q[S, :]
    prob = softmax_np(logits)
    Y = sample_categorical(prob, rng)

    return X.astype(np.float32), S.astype(np.int64), Y.astype(np.int64)


# ----------------------------
# Models
# ----------------------------

class SourceMultinomialModel(nn.Module):
    def __init__(self, K, C, P, mode="pooled", m=None):
        super().__init__()
        self.K = K
        self.C = C
        self.P = P
        self.mode = mode
        self.m = m

        self.alpha = nn.Parameter(torch.zeros(C))
        self.beta = nn.Parameter(0.05 * torch.randn(C, P))

        if mode == "One-Hot Encoding":
            self.theta = nn.Parameter(0.01 * torch.randn(K, C))

        elif mode == "embedding":
            if m is None:
                raise ValueError("Embedding dimension m must be provided.")
            self.E = nn.Parameter(0.05 * torch.randn(K, m))
            self.A = nn.Parameter(0.05 * torch.randn(C, m))

        elif mode == "pooled":
            pass

        else:
            raise ValueError("mode must be 'pooled', 'One-Hot Encoding', or 'embedding'.")

    def source_matrix(self):
        if self.mode == "pooled":
            return torch.zeros(self.K, self.C, device=self.alpha.device)

        if self.mode == "One-Hot Encoding":
            Q = self.theta
            return Q - Q.mean(dim=0, keepdim=True)

        if self.mode == "embedding":
            Q = self.E @ self.A.T
            return Q - Q.mean(dim=0, keepdim=True)

    def penalty(self):
        if self.mode == "pooled":
            return torch.tensor(0.0, device=self.alpha.device)

        if self.mode == "One-Hot Encoding":
            Q = self.source_matrix()
            return (Q ** 2).mean()

        if self.mode == "embedding":
            return (self.E ** 2).mean() + (self.A ** 2).mean()

    def forward(self, X, S):
        logits = X @ self.beta.T + self.alpha

        if self.mode == "pooled":
            return logits

        Q = self.source_matrix()
        return logits + Q[S, :]


def fit_model(X, S, Y, K, mode, m=None, lambda_grid=LAMBDA_GRID, seed=1):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    tr_idx, val_idx = train_val_split(n, VAL_FRAC, rng)

    X_t = torch.tensor(X, device=DEVICE)
    S_t = torch.tensor(S, device=DEVICE)
    Y_t = torch.tensor(Y, device=DEVICE)

    X_tr, S_tr, Y_tr = X_t[tr_idx], S_t[tr_idx], Y_t[tr_idx]
    X_val, S_val, Y_val = X_t[val_idx], S_t[val_idx], Y_t[val_idx]

    best_state = None
    best_val = np.inf
    best_lam = None

    for lam in lambda_grid:
        torch.manual_seed(seed + int(1e6 * lam))
        model = SourceMultinomialModel(K, C, P, mode=mode, m=m).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        best_val_lam = np.inf
        best_state_lam = None
        bad_epochs = 0

        for epoch in range(EPOCHS):
            model.train()
            opt.zero_grad()

            logits = model(X_tr, S_tr)
            loss = F.cross_entropy(logits, Y_tr) + lam * model.penalty()
            loss.backward()
            opt.step()

            model.eval()
            with torch.no_grad():
                val_logits = model(X_val, S_val)
                val_loss = F.cross_entropy(val_logits, Y_val).item()

            if val_loss < best_val_lam - 1e-5:
                best_val_lam = val_loss
                best_state_lam = copy.deepcopy(model.state_dict())
                bad_epochs = 0
            else:
                bad_epochs += 1

            if bad_epochs >= PATIENCE:
                break

        if best_val_lam < best_val:
            best_val = best_val_lam
            best_lam = lam
            best_state = best_state_lam

    final_model = SourceMultinomialModel(K, C, P, mode=mode, m=m).to(DEVICE)
    final_model.load_state_dict(best_state)
    final_model.eval()

    return final_model, best_lam, best_val


def evaluate_model(model, X_test, S_test, Y_test, Q_true):
    X_t = torch.tensor(X_test, device=DEVICE)
    S_t = torch.tensor(S_test, device=DEVICE)
    Y_t = torch.tensor(Y_test, device=DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_t, S_t)
        logloss = F.cross_entropy(logits, Y_t).item()

        Q_hat = model.source_matrix().detach().cpu().numpy()

    see = np.mean((Q_hat - Q_true) ** 2)

    return logloss, see


# ----------------------------
# Simulation runners
# ----------------------------

def run_single_configuration(K, ns, regime, m, rep_seed):
    rng = np.random.default_rng(rep_seed)

    alpha, beta, Q = generate_parameters(K, regime, rng)

    X, S, Y = sample_dataset(
        K=K,
        n_per_source=ns,
        alpha=alpha,
        beta=beta,
        Q=Q,
        rng=rng,
        balanced=True
    )

    X_test, S_test, Y_test = sample_dataset(
        K=K,
        n_per_source=N_TEST,
        alpha=alpha,
        beta=beta,
        Q=Q,
        rng=rng,
        balanced=False
    )

    rows = []

    # Pooled
    model, lam, _ = fit_model(X, S, Y, K, mode="pooled", m=None, seed=rep_seed + 11)
    logloss, see = evaluate_model(model, X_test, S_test, Y_test, Q)
    rows.append({
        "model": "Pooled",
        "K": K,
        "ns": ns,
        "m": np.nan,
        "regime": regime,
        "logloss": logloss,
        "see": see,
        "lambda": lam,
        "rep": rep_seed
    })

    # Dummy
    model, lam, _ = fit_model(X, S, Y, K, mode="One-Hot Encoding", m=None, seed=rep_seed + 22)
    logloss, see = evaluate_model(model, X_test, S_test, Y_test, Q)
    rows.append({
        "model": "One-Hot Encoding",
        "K": K,
        "ns": ns,
        "m": np.nan,
        "regime": regime,
        "logloss": logloss,
        "see": see,
        "lambda": lam,
        "rep": rep_seed
    })

    # Embedding
    model, lam, _ = fit_model(X, S, Y, K, mode="embedding", m=m, seed=rep_seed + 33)
    logloss, see = evaluate_model(model, X_test, S_test, Y_test, Q)
    rows.append({
        "model": f"Embedding",
        "K": K,
        "ns": ns,
        "m": m,
        "regime": regime,
        "logloss": logloss,
        "see": see,
        "lambda": lam,
        "rep": rep_seed
    })

    return rows


def run_figure_1a_simulation():
    """
    Figure 1(a): log-loss versus K.
    Embedding dimension is fixed at m=4, the true rank in the low-rank regime.
    """
    all_rows = []

    for regime in REGIMES:
        for ns in NS_GRID:
            for K in K_GRID:
                for r in range(N_REPS):
                    seed = 100000 + 1000 * r + 10 * K + ns + (0 if regime == "low_rank" else 500000)
                    print(f"[Fig1a] regime={regime}, ns={ns}, K={K}, rep={r+1}/{N_REPS}")
                    all_rows.extend(
                        run_single_configuration(
                            K=K,
                            ns=ns,
                            regime=regime,
                            m=4,
                            rep_seed=seed
                        )
                    )

    df = pd.DataFrame(all_rows)
    df.to_csv(os.path.join(OUTDIR, "source_complexity_logloss_results.csv"), index=False)
    return df


def run_figure_1b_simulation():
    """
    Figure 1(b): embedding dimension effect.
    We fix K=50 and vary m.
    """
    all_rows = []
    K_FIXED = 50

    for regime in REGIMES:
        for ns in NS_GRID:
            for m in M_GRID:
                for r in range(N_REPS):
                    seed = 200000 + 1000 * r + 100 * m + ns + (0 if regime == "low_rank" else 500000)
                    print(f"[Fig1b] regime={regime}, ns={ns}, m={m}, rep={r+1}/{N_REPS}")

                    # Only embedding is needed for dimension-effect plot.
                    rng = np.random.default_rng(seed)
                    alpha, beta, Q = generate_parameters(K_FIXED, regime, rng)

                    X, S, Y = sample_dataset(
                        K=K_FIXED,
                        n_per_source=ns,
                        alpha=alpha,
                        beta=beta,
                        Q=Q,
                        rng=rng,
                        balanced=True
                    )

                    X_test, S_test, Y_test = sample_dataset(
                        K=K_FIXED,
                        n_per_source=N_TEST,
                        alpha=alpha,
                        beta=beta,
                        Q=Q,
                        rng=rng,
                        balanced=False
                    )

                    model, lam, _ = fit_model(
                        X, S, Y, K_FIXED,
                        mode="embedding",
                        m=m,
                        seed=seed + 33
                    )
                    logloss, see = evaluate_model(model, X_test, S_test, Y_test, Q)

                    all_rows.append({
                        "model": "Embedding",
                        "K": K_FIXED,
                        "ns": ns,
                        "m": m,
                        "regime": regime,
                        "logloss": logloss,
                        "see": see,
                        "lambda": lam,
                        "rep": seed
                    })

    df = pd.DataFrame(all_rows)
    df.to_csv(os.path.join(OUTDIR, "source_complexity_embedding_dim_results.csv"), index=False)
    return df


# ----------------------------
# Plotting
# ----------------------------

def summarize(df, group_cols, metric):
    out = (
        df.groupby(group_cols)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    out["se"] = out["std"] / np.sqrt(out["count"])
    return out


def regime_label(regime):
    return "Low-rank source effects" if regime == "low_rank" else "Unrestricted source effects"


def plot_figure_1a(df):
    summary = summarize(
        df,
        group_cols=["regime", "ns", "K", "model"],
        metric="logloss"
    )

    colors = {
        "Pooled": "#4D4D4D",
        "One-Hot Encoding": "#0072B2",
        "Embedding": "#D55E00"
    }

    fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.2), sharex=True, sharey=False)

    panel_order = [
        ("low_rank", 40),
        ("low_rank", 200),
        ("unrestricted", 40),
        ("unrestricted", 200)
    ]

    for ax, (regime, ns) in zip(axes.ravel(), panel_order):
        sub = summary[(summary["regime"] == regime) & (summary["ns"] == ns)]

        for model in ["Pooled", "One-Hot Encoding", "Embedding"]:
            sm = sub[sub["model"] == model].sort_values("K")
            ax.errorbar(
                sm["K"],
                sm["mean"],
                yerr=sm["se"],
                marker="o",
                linewidth=2,
                capsize=3,
                label=model,
                color=colors[model]
            )

        ax.set_title(f"{regime_label(regime)}, $n_s={ns}$")
        ax.set_xlabel("Number of sources $K$")
        ax.set_ylabel("Test log-loss")
        ax.grid(alpha=0.25)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.94])

    path = os.path.join(OUTDIR, "figure_source_complexity_logloss.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {path}")


def plot_figure_1b(df):
    s_log = summarize(
        df,
        group_cols=["regime", "ns", "m"],
        metric="logloss"
    )
    s_see = summarize(
        df,
        group_cols=["regime", "ns", "m"],
        metric="see"
    )

    colors = {
        40: "#D55E00",
        200: "#0072B2"
    }

    fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.2), sharex=True)

    for row, regime in enumerate(REGIMES):
        # Log-loss
        ax = axes[row, 0]
        sub = s_log[s_log["regime"] == regime]
        for ns in NS_GRID:
            sm = sub[sub["ns"] == ns].sort_values("m")
            ax.errorbar(
                sm["m"],
                sm["mean"],
                yerr=sm["se"],
                marker="o",
                linewidth=2,
                capsize=3,
                label=f"$n_s={ns}$",
                color=colors[ns]
            )
        ax.axvline(R0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{regime_label(regime)}: test log-loss")
        ax.set_ylabel("Test log-loss")
        ax.grid(alpha=0.25)

        # SEE
        ax = axes[row, 1]
        sub = s_see[s_see["regime"] == regime]
        for ns in NS_GRID:
            sm = sub[sub["ns"] == ns].sort_values("m")
            ax.errorbar(
                sm["m"],
                sm["mean"],
                yerr=sm["se"],
                marker="o",
                linewidth=2,
                capsize=3,
                label=f"$n_s={ns}$",
                color=colors[ns]
            )
        ax.axvline(R0, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"{regime_label(regime)}: source-effect error")
        ax.set_ylabel(r"Source-effect error")
        ax.grid(alpha=0.25)

    for ax in axes[-1, :]:
        ax.set_xlabel("Embedding dimension $m$")
        ax.set_xticks(M_GRID)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.94])

    path = os.path.join(OUTDIR, "figure_source_complexity_embedding_dim.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {path}")


# ----------------------------
# Main
# ----------------------------

if __name__ == "__main__":
    print(f"Using device: {DEVICE}")
    df_a = run_figure_1a_simulation()
    plot_figure_1a(df_a)

    #df_b = run_figure_1b_simulation()
    #plot_figure_1b(df_b)

    print("Done.")


# ============================================================
# experiment 2: covariance DGP
# ============================================================

@dataclass
class CovarianceDGP:
    n: int = 6000
    p: int = 4
    K: int = 4
    C: int = 3
    m: int = 3
    delta: float = 1.0
    noise_sd: float = 1.0
    seed: int = 456


def generate_covariance_data(dgp: CovarianceDGP) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(dgp.seed)

    Y = rng.choice(dgp.C, size=dgp.n)
    S = rng.choice(dgp.K, size=dgp.n)

    U = rng.normal(0, 1.0, size=(dgp.C, dgp.p))  # class-specific locations
    E = rng.normal(0, 1.0, size=(dgp.K, dgp.m))  # source embeddings
    G = rng.normal(0, 1.0 / np.sqrt(dgp.m), size=(dgp.m, dgp.p))  # loading matrix
    eps = rng.normal(0, dgp.noise_sd, size=(dgp.n, dgp.p))

    H = U[Y] + dgp.delta * (E[S] @ G) + eps
    return H, Y, S


def compute_covariance_quantities(H: np.ndarray, Y: np.ndarray, S: np.ndarray) -> Dict[str, float]:
    pooled_trace = float(np.trace(pooled_within_class_covariance(H, Y)))

    n = len(Y)
    classes = np.unique(Y)
    sources = np.unique(S)
    p = H.shape[1]
    sigma_cond = np.zeros((p, p))

    for c in classes:
        for s in sources:
            idx = (Y == c) & (S == s)
            hcs = H[idx]
            if hcs.shape[0] <= 1:
                continue
            weight = hcs.shape[0] / n
            centered = hcs - hcs.mean(axis=0, keepdims=True)
            cov_cs = centered.T @ centered / hcs.shape[0]
            sigma_cond += weight * cov_cs

    cond_trace = float(np.trace(sigma_cond))

    return {
        "pooled_within_trace": pooled_trace,
        "source_conditioned_within_trace": cond_trace,
        "source_induced_inflation": pooled_trace - cond_trace,
    }


# ============================================================
# experiment 3: regression DGP
# ============================================================

@dataclass
class OVBdgp:
    n: int = 5000
    p: int = 5
    K: int = 6
    m: int = 4
    delta: float = 1.0
    beta_scale: float = 1.0
    x_s_dependence: float = 0.7
    noise_sd: float = 1.0
    seed: int = 789


def generate_ovb_data(dgp: OVBdgp) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(dgp.seed)

    S = rng.choice(dgp.K, size=dgp.n)

    E0 = rng.normal(0, 1.0, size=(dgp.K, dgp.m))
    Bz = rng.normal(0, dgp.x_s_dependence / np.sqrt(dgp.m), size=(dgp.m, dgp.p))
    zeta = rng.normal(0, 1.0, size=(dgp.n, dgp.p))
    Z = zeta + E0[S] @ Bz

    beta0 = rng.normal(0, dgp.beta_scale, size=dgp.p)
    a0 = rng.normal(0, 1.0, size=dgp.m)
    eps = rng.normal(0, dgp.noise_sd, size=dgp.n)

    q0 = dgp.delta * (E0 @ a0)
    T = Z @ beta0 + q0[S] + eps

    return Z, S, T, beta0


# ============================================================
# trainable source-aware models
# ============================================================

class SourceAwareMulticlass(nn.Module):
    """
    eta_c(X, S) = linear_X_c(X) + <e_S, a_c>
    where E = [e_1, ..., e_K]^T is trainable.
    """
    def __init__(self, p: int, K: int, C: int, m: int):
        super().__init__()
        self.linear = nn.Linear(p, C)
        self.embedding = nn.Embedding(K, m)
        self.A = nn.Parameter(torch.zeros(m, C))

        nn.init.normal_(self.linear.weight, mean=0.0, std=0.05)
        nn.init.zeros_(self.linear.bias)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.1)
        nn.init.normal_(self.A, mean=0.0, std=0.1)

    def forward(self, x: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        base = self.linear(x)
        e = self.embedding(s)                # (batch, m)
        source_term = e @ self.A            # (batch, C)
        return base + source_term


class SourceAwareLinear(nn.Module):
    """
    eta(Z, S) = beta^T Z + <e_S, a>
    with jointly trainable beta, E, a.
    """
    def __init__(self, p: int, K: int, m: int):
        super().__init__()
        self.linear = nn.Linear(p, 1)
        self.embedding = nn.Embedding(K, m)
        self.a = nn.Parameter(torch.zeros(m, 1))

        nn.init.normal_(self.linear.weight, mean=0.0, std=0.05)
        nn.init.zeros_(self.linear.bias)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.1)
        nn.init.normal_(self.a, mean=0.0, std=0.1)

    def forward(self, z: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        base = self.linear(z)                     # (batch, 1)
        e = self.embedding(s)                     # (batch, m)
        source_term = e @ self.a                  # (batch, 1)
        return base + source_term


def fit_source_aware_multiclass(
    X_train: np.ndarray,
    S_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    S_test: np.ndarray,
    p: int,
    K: int,
    C: int,
    m: int,
    seed: int,
    lr: float = 1e-2,
    weight_decay: float = 1e-4,
    batch_size: int = 256,
    epochs: int = 50,
) -> Tuple[np.ndarray, np.ndarray]:
    set_all_seeds(seed)

    model = SourceAwareMulticlass(p=p, K=K, C=C, m=m).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    dataset = TensorDataset(
        torch.tensor(X_train, dtype=TORCH_DTYPE),
        torch.tensor(S_train, dtype=torch.long),
        torch.tensor(y_train, dtype=torch.long),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for _ in range(epochs):
        for xb, sb, yb in loader:
            xb = xb.to(DEVICE)
            sb = sb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb, sb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        xt = torch.tensor(X_test, dtype=TORCH_DTYPE, device=DEVICE)
        st = torch.tensor(S_test, dtype=torch.long, device=DEVICE)
        logits = model(xt, st).cpu().numpy()
        probs = softmax_np(logits)
        pred = probs.argmax(axis=1)

    return pred, probs


def fit_source_aware_linear(
    Z_train: np.ndarray,
    S_train: np.ndarray,
    T_train: np.ndarray,
    Z_test: np.ndarray,
    S_test: np.ndarray,
    p: int,
    K: int,
    m: int,
    seed: int,
    lr: float = 1e-2,
    weight_decay: float = 1e-4,
    batch_size: int = 256,
    epochs: int = 200,
) -> Tuple[np.ndarray, np.ndarray]:
    set_all_seeds(seed)

    model = SourceAwareLinear(p=p, K=K, m=m).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    dataset = TensorDataset(
        torch.tensor(Z_train, dtype=TORCH_DTYPE),
        torch.tensor(S_train, dtype=torch.long),
        torch.tensor(T_train.reshape(-1, 1), dtype=TORCH_DTYPE),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for _ in range(epochs):
        for zb, sb, tb in loader:
            zb = zb.to(DEVICE)
            sb = sb.to(DEVICE)
            tb = tb.to(DEVICE)

            optimizer.zero_grad()
            pred = model(zb, sb)
            loss = criterion(pred, tb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        zt = torch.tensor(Z_test, dtype=TORCH_DTYPE, device=DEVICE)
        st = torch.tensor(S_test, dtype=torch.long, device=DEVICE)
        pred_test = model(zt, st).cpu().numpy().reshape(-1)

        beta_hat = model.linear.weight.detach().cpu().numpy().reshape(-1)

    return pred_test, beta_hat


# ============================================================
# experiment runners
# ============================================================

def run_classification_experiment(
    n_reps: int = 100,
    delta_grid: List[float] = (0.0, 0.25, 0.5, 1.0, 1.5),
    base_seed: int = 1000,
) -> pd.DataFrame:
    rows = []

    K1 = 5
    m1 = 2
    p1 = 6
    C1 = 3

    for delta in delta_grid:
        for rep in range(n_reps):
            seed = base_seed + 10000 * int(100 * delta) + rep
            dgp = ClassificationDGP(n=4000, p=p1, K=K1, C=C1, m=m1, delta=delta, seed=seed)
            X, S, Y = generate_source_aware_classification(dgp)

            X_train, X_test, S_train, S_test, Y_train, Y_test = train_test_split(
                X, S, Y, test_size=0.4, random_state=seed, stratify=Y
            )

            # source-agnostic pooled model
            pooled = LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=2000,
                random_state=seed,
            )
            pooled.fit(X_train, Y_train)
            pred_x = pooled.predict(X_test)
            prob_x = pooled.predict_proba(X_test)

            # jointly trainable source-aware model
            pred_xs, prob_xs = fit_source_aware_multiclass(
                X_train, S_train, Y_train,
                X_test, S_test,
                p=p1, K=K1, C=C1, m=m1, seed=seed,
            )

            err_x = 1.0 - accuracy_score(Y_test, pred_x)
            err_xs = 1.0 - accuracy_score(Y_test, pred_xs)

            rows.append({
                "rep": rep,
                "delta": delta,
                "K": K1,
                "m": m1,
                "err_x": err_x,
                "err_xs": err_xs,
                "logloss_x": log_loss(Y_test, prob_x),
                "logloss_xs": log_loss(Y_test, prob_xs),
                "improvement_error": err_x - err_xs,
                "improvement_logloss": log_loss(Y_test, prob_x) - log_loss(Y_test, prob_xs),
            })

    return pd.DataFrame(rows)


def run_covariance_experiment(
    n_reps: int = 100,
    delta_grid: List[float] = (0.0, 0.25, 0.5, 1.0, 1.5),
    base_seed: int = 2000,
) -> pd.DataFrame:
    rows = []

    K2 = 4
    m2 = 3

    for delta in delta_grid:
        for rep in range(n_reps):
            seed = base_seed + 10000 * int(100 * delta) + rep
            dgp = CovarianceDGP(n=6000, p=4, K=K2, C=3, m=m2, delta=delta, seed=seed)
            H, Y, S = generate_covariance_data(dgp)
            out = compute_covariance_quantities(H, Y, S)
            out["rep"] = rep
            out["delta"] = delta
            out["K"] = K2
            out["m"] = m2
            rows.append(out)

    return pd.DataFrame(rows)


def run_ovb_experiment(
    n_reps: int = 150,
    delta_grid: List[float] = (0.0, 0.5, 1.0, 1.5),
    base_seed: int = 3000,
) -> pd.DataFrame:
    rows = []

    K3 = 6
    m3 = 4
    p3 = 5

    for delta in delta_grid:
        for rep in range(n_reps):
            seed = base_seed + 10000 * int(100 * delta) + rep
            dgp = OVBdgp(n=5000, p=p3, K=K3, m=m3, delta=delta, seed=seed)
            Z, S, T, beta0 = generate_ovb_data(dgp)

            # source-omitted baseline
            omit = LinearRegression()
            omit.fit(Z, T)
            beta_tilde = omit.coef_
            resid_omit = T - omit.predict(Z)

            # jointly trainable source-aware model
            pred_full, beta_full = fit_source_aware_linear(
                Z, S, T, Z, S,
                p=p3, K=K3, m=m3, seed=seed,
            )
            resid_full = T - pred_full

            rows.append({
                "rep": rep,
                "delta": delta,
                "K": K3,
                "m": m3,
                "beta_bias_norm_omit": np.linalg.norm(beta_tilde - beta0),
                "beta_bias_norm_full": np.linalg.norm(beta_full - beta0),
                "resid_var_omit": np.var(resid_omit),
                "resid_var_full": np.var(resid_full),
                "resid_var_inflation": np.var(resid_omit) - np.var(resid_full),
            })

    return pd.DataFrame(rows)


# ============================================================
# plotting
# ============================================================

def plot_prediction_figure(df_cls: pd.DataFrame) -> None:
    summ = summarize_with_ci(df_cls, "delta", ["err_x", "err_xs", "improvement_error"])
    x = summ["delta"].values

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        x, summ["err_x_mean"], yerr=summ["err_x_ci"],
        marker="o", linewidth=2, capsize=4, label="Source-agnostic"
    )
    ax.errorbar(
        x, summ["err_xs_mean"], yerr=summ["err_xs_ci"],
        marker="s", linewidth=2, capsize=4, label="Source-aware"
    )
    ax.set_xlabel("Source-shift magnitude")
    ax.set_ylabel("Test classification error")
    ax.set_title("Prediction error under increasing source heterogeneity")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("figure2.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        x, summ["improvement_error_mean"], yerr=summ["improvement_error_ci"],
        marker="o", linewidth=2, capsize=4
    )
    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel("Source-shift magnitude")
    ax.set_ylabel("Error reduction from source awareness")
    ax.set_title("Performance gain from incorporating source information")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("figure3.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_covariance_figure(df_cov: pd.DataFrame) -> None:
    summ = summarize_with_ci(
        df_cov,
        "delta",
        ["pooled_within_trace", "source_conditioned_within_trace", "source_induced_inflation"]
    )
    x = summ["delta"].values

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        x, summ["pooled_within_trace_mean"], yerr=summ["pooled_within_trace_ci"],
        marker="o", linewidth=2, capsize=4, label="Ignoring source"
    )
    ax.errorbar(
        x, summ["source_conditioned_within_trace_mean"], yerr=summ["source_conditioned_within_trace_ci"],
        marker="s", linewidth=2, capsize=4, label="Conditioning on source"
    )
    ax.set_xlabel("Source-shift magnitude")
    ax.set_ylabel("Trace of within-class covariance")
    ax.set_title("Within-class covariance inflation from source mixing")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("figure4.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        x, summ["source_induced_inflation_mean"], yerr=summ["source_induced_inflation_ci"],
        marker="o", linewidth=2, capsize=4
    )
    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel("Source-shift magnitude")
    ax.set_ylabel("Source-induced covariance inflation")
    ax.set_title("Additional within-class variability due to source heterogeneity")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("figure1.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_ovb_figure(df_ovb: pd.DataFrame) -> None:
    summ = summarize_with_ci(
        df_ovb,
        "delta",
        ["beta_bias_norm_omit", "beta_bias_norm_full", "resid_var_inflation"]
    )
    x = summ["delta"].values

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        x, summ["beta_bias_norm_omit_mean"], yerr=summ["beta_bias_norm_omit_ci"],
        marker="o", linewidth=2, capsize=4, label="Omitting source"
    )
    ax.errorbar(
        x, summ["beta_bias_norm_full_mean"], yerr=summ["beta_bias_norm_full_ci"],
        marker="s", linewidth=2, capsize=4, label="Source-aware"
    )
    ax.set_xlabel("Magnitude of source effect")
    ax.set_ylabel(r"$\|\hat{\beta}-\beta_0\|_2$")
    ax.set_title("Coefficient bias under source omission")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("figure5.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(
        x, summ["resid_var_inflation_mean"], yerr=summ["resid_var_inflation_ci"],
        marker="o", linewidth=2, capsize=4
    )
    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_xlabel("Magnitude of source effect")
    ax.set_ylabel("Residual variance inflation")
    ax.set_title("Residual inflation induced by omitting source information")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("figure6.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# main
# ============================================================

if __name__ == "__main__":
    print(f"Using device: {DEVICE}")

    df_cls = run_classification_experiment(
        n_reps=100,
        delta_grid=[0.0, 0.25, 0.5, 1.0, 1.5],
    )
    df_cov = run_covariance_experiment(
        n_reps=100,
        delta_grid=[0.0, 0.25, 0.5, 1.0, 1.5],
    )
    df_ovb = run_ovb_experiment(
        n_reps=150,
        delta_grid=[0.0, 0.5, 1.0, 1.5],
    )

    df_cls.to_csv("section_3_1_classification_results.csv", index=False)
    df_cov.to_csv("section_3_1_covariance_results.csv", index=False)
    df_ovb.to_csv("section_3_1_ovb_results.csv", index=False)

    plot_prediction_figure(df_cls)
    plot_covariance_figure(df_cov)
    plot_ovb_figure(df_ovb)

    print("Saved:")
    print("  section_3_1_classification_results.csv")
    print("  section_3_1_covariance_results.csv")
    print("  section_3_1_ovb_results.csv")
    print("  figure1.png")
    print("  figure2.png")
    print("  figure3.png")
    print("  figure4.png")
    print("  figure5.png")
    print("  figure6.png")
    print()
    print("Experiment-specific source counts and embedding dimensions:")
    print("  Experiment 1: K = 5, m = 2")
    print("  Experiment 2: K = 4, m = 3")
    print("  Experiment 3: K = 6, m = 4")

Using device: cpu
[Fig1a] regime=low_rank, ns=40, K=10, rep=1/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=2/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=3/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=4/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=5/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=6/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=7/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=8/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=9/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=10/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=11/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=12/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=13/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=14/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=15/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=16/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=17/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=18/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=19/100
[Fig1a] regime=low_rank, ns=40, K=10, rep=20/100
[Fig1a] reg

NameError: name 'ClassificationDGP' is not defined